### 5. ✅ Rate Limiter — (API / Django REST / Redis context)
**Why it impresses**: **sliding window** algorithm, you can pivot to:

> "in production I'd use Redis with `INCR`/`EXPIRE`" — shows systems thinking.

In [ ]:
import functools
import time
from collections import defaultdict, deque

def rate_limit(max_calls: int, period: float):
    """
    Sliding window rate limiter.
    Real use case: protecting API endpoints, preventing abuse.
    In production: use Redis for distributed rate limiting across workers.
    """
    call_timestamps: dict[str, deque] = defaultdict(deque)

    def decorator(func: Callable):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            key = func.__name__  # In real API: use user_id or IP
            now = time.monotonic()
            timestamps = call_timestamps[key]

            # Evict timestamps outside the sliding window
            while timestamps and timestamps[0] < now - period:
                timestamps.popleft()

            if len(timestamps) >= max_calls:
                wait = period - (now - timestamps[0])
                raise Exception(
                    f"Rate limit exceeded for '{key}'. "
                    f"Retry after {wait:.2f}s."
                )

            timestamps.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator


# --- Usage ---
@rate_limit(max_calls=3, period=10.0)  # 3 calls per 10 seconds
def send_email(recipient: str, body: str):
    print(f"Email sent to {recipient}")

# Simulating rapid calls
for i in range(5):
    try:
        send_email("user@example.com", f"Message {i}")
    except Exception as e:
        print(f"Call {i+1}: {e}")